**Simple RAG Application:** Turn the Week 7 RAG flow into a small usable app, and add
basic error handling.

In [1]:
%pip install chromadb sentence-transformers google-genai python-dotenv

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import time
import chromadb
from sentence_transformers import SentenceTransformer
from google import genai
from google.genai.errors import APIError
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise SystemExit("No GEMINI_API_KEY found.")

client = genai.Client(api_key=api_key)
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Connect to the SAME database
db_client = chromadb.PersistentClient(path="./week8_rag_db")
collection = db_client.get_or_create_collection(name="week8_documents")

print("App is ready. Documents currently stored:", collection.count())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

App is ready. Documents currently stored: 6


**Retrieval, with error handling**

`distance` is a number ChromaDB gives us for every result: the SMALLER the distance, the more
similar the chunk is to the question. We use a `max_distance` cutoff to decide "this chunk is
not actually related, so do not use it as context".

In [3]:
def retrieve_context(question, top_k=3, max_distance=1.1):
    """
    Looks for the most relevant chunks for a question.
    Returns a context string, OR None if nothing relevant was found.
    """
    # Error handling #2: an empty database means there is nothing to search.
    if collection.count() == 0:
        return None

    question_embedding = embedding_model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=top_k
    )

    chunks = results["documents"][0]
    distances = results["distances"][0]

    # Error handling #3: if even the closest chunk is too different,
    # treat it as "nothing relevant found" rather than forcing a bad context.
    if not chunks or min(distances) > max_distance:
        return None

    context = "\n".join(
        f"Context {i}: {chunk}"
        for i, chunk in enumerate(chunks, start=1)
    )

    return context

Asking Gemini, with retries

In [ ]:
def ask_gemini(question, context=None):
    if context:
        prompt = f"""
Answer the question using the retrieved context when it is relevant.
If the context does not contain the answer, use your general knowledge.

Retrieved context:
{context}

Question:
{question}

Give a concise and accurate answer.
"""
    else:
        prompt = f"""
No relevant information was found in the user's documents for this question.
Answer using your general knowledge, and briefly mention that this answer is
not based on the user's own documents.

Question:
{question}
"""
    model = "gemini-3.6-flash"
    for attempt in range(5):
        try:
            response = client.models.generate_content(model=model, contents=prompt)
            if response.text:
                return response.text.strip()
            return "Gemini returned an empty answer."

        except APIError as error:
            error_text = str(error)
            if any(code in error_text for code in ("503", "UNAVAILABLE", "429", "RESOURCE_EXHAUSTED")):
                    wait_seconds = 2 ** attempt
                    print(f"{model} is busy. Retrying in {wait_seconds} seconds...")
                    time.sleep(wait_seconds)
            else:
                # Error handling #4: any other API error is reported
                return f"Gemini API error: {error}"

    return "gemini-3.6-flash is temporarily unavailable. Please try again later."

## Putting it together: one "ask a question" step

In [5]:
def answer_question(question):
    """
    Runs the full RAG flow for one question and prints the result.
    Wrapped in try/except so one bad question never crashes the app.
    """
    try:
        context = retrieve_context(question)

        if context is None:
            print("(No matching information was found in your documents.)")
        else:
            print("\nRetrieved context:")
            print(context)

        answer = ask_gemini(question, context)
        print("\nAnswer:")
        print(answer)

    except Exception as error:
        # A final safety net: catches anything unexpected (bad internet
        # connection, a typo in our own code, etc.) and shows a clean message.
        print(f"Something went wrong while answering this question: {error}")

**The app loop**

`input()` pauses the notebook and waits for you to type something and press Enter.
Type `exit` or `quit` to stop the loop.

In [6]:
def run_app():
    print("Simple RAG App. Type your question, or type 'exit' to stop.\n")

    while True:
        question = input("Your question: ").strip()

        # Error handling #5: an empty question (user just pressed Enter).
        if question == "":
            print("Please type a question, or 'exit' to stop.\n")
            continue

        if question.lower() in ("exit", "quit"):
            print("Goodbye!")
            break

        print("=" * 70)
        print("QUESTION:", question)
        answer_question(question)
        print("=" * 70, "\n")